In [0]:
from pyspark.sql import functions as F

#Reading from bronze table

In [0]:
items_df = spark.read.table("albion_project.bronze.items")
display(items_df)

#Checking data

In [0]:
items_df.printSchema()
items_df.count()
items_df.select("item_id").distinct().count()

In [0]:
items_df.filter(F.col("item_id").isNull()).count()

In [0]:
items_df.filter(F.col("item_name_en").isNull()).count()
items_df.filter(F.col("item_name_pl").isNull()).count()

#Trimming names

In [0]:
items_silver_df = (
    items_df
    .withColumn("item_id", F.trim("item_id"))
    .withColumn("item_name_en", F.trim("item_name_en"))
    .withColumn("item_name_pl", F.trim("item_name_pl"))
)

#Adding display_name in case of empty names

In [0]:
items_silver_df = (
    items_silver_df
    .withColumn(
        "display_name_en",
        F.coalesce(
            F.col("item_name_en"),
            F.col("item_id")
        )
    )
    .withColumn(
        "display_name_pl",
        F.coalesce(
            F.col("item_name_pl"),
            F.col("item_id")
        )
    )
)

#Writing to silver table

In [0]:
(
    items_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("albion_project.silver.items")
)